# Walkthrough: linear_regression.ipynb

**Statistisches Lernen 2 — FH Kufstein Tirol**  
Arquivo de referência: `linear_regression.ipynb`

Este arquivo explica **passo a passo** o que foi feito no notebook original, **por que** cada decisão foi tomada, **como cada ferramenta funciona**, e como você deve proceder ao usá-las em situações similares.

---

> **Regra geral deste walkthrough:**  
> Para cada célula do original — O que o código faz → Por que foi feito assim → Como a ferramenta funciona → Quando usar / Como adaptar

---

## Visão geral do notebook original

O `linear_regression.ipynb` implementa **Linear Regression** do zero usando apenas álgebra linear — sem scikit-learn, sem fórmulas mágicas. O objetivo é mostrar que a regressão linear é, na sua essência, a solução de um sistema de equações.

| Etapa | O que acontece |
|-------|----------------|
| 1 | Criar os dados $(x, y)$ como arrays NumPy |
| 2 | Construir a **Design Matrix** $X$ |
| 3 | Montar as **Normal Equations** |
| 4 | Resolver o sistema para encontrar os coeficientes $a$ |
| 5 | Gerar previsões e plotar o resultado |

---

## Passo 1 — Importar NumPy e criar os dados

### O que o código faz

```python
import numpy as np

x = np.array([-7, -4, -2, 0, 2])
y = np.array([ 1,  2,  2, 0, -1])
```

Cria dois arrays unidimensionais: `x` (as entradas, as features) e `y` (as saídas, o target). São os 5 pontos de dados que o modelo vai aprender a aproximar.

---

### Como a ferramenta funciona: `numpy` e `np.array`

**NumPy** é a biblioteca de computação numérica do Python. Um `np.array` é basicamente uma lista, mas:
- Muito mais rápido (operações em C por baixo)
- Suporta operações vetorizadas (sem `for` loops): `x * 2`, `x + y`, `x @ y`
- Suporta álgebra linear: multiplicação de matrizes, inversão, decomposição

**Anatomia de um array:**
```
np.array([1, 2, 3])   → shape (3,)    — vetor 1D (linha)
np.array([[1, 2],
          [3, 4]])    → shape (2, 2)  — matriz 2D
```

### Por que foi feito assim

Os dados foram definidos manualmente para manter o exemplo pequeno e verificável à mão. Em projetos reais, os dados viriam de um arquivo CSV, banco de dados, ou API.

### Como proceder em situações reais

```python
import pandas as pd
import numpy as np

df = pd.read_csv("meus_dados.csv")
x = df["feature_coluna"].to_numpy()   # converter Series para np.array
y = df["target_coluna"].to_numpy()
```

**Regra de ouro:** sempre verifique o `shape` depois de criar arrays:
```python
print(x.shape)   # (5,) → vetor com 5 elementos
print(y.shape)   # (5,) → vetor com 5 elementos
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Os mesmos dados do notebook original
x = np.array([-7, -4, -2, 0, 2])
y = np.array([ 1,  2,  2, 0, -1])

print("Dados carregados:")
print(f"  x: {x}  |  shape: {x.shape}  |  dtype: {x.dtype}")
print(f"  y: {y}  |  shape: {y.shape}  |  dtype: {y.dtype}")
print(f"  N = {len(x)} pontos de dados")

# Visualizar os dados antes de qualquer fitting
plt.figure(figsize=(7, 4))
plt.scatter(x, y, c="red", s=80, zorder=3, label="dados (x, y)")
for xi, yi in zip(x, y):
    plt.annotate(f"({xi}, {yi})", (xi, yi), textcoords="offset points",
                 xytext=(5, 8), fontsize=9)
plt.axhline(0, color="gray", lw=0.8, ls="--")
plt.axvline(0, color="gray", lw=0.8, ls="--")
plt.title("Passo 1 — Os dados brutos")
plt.xlabel("x (feature)"); plt.ylabel("y (target)")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---

## Passo 2 — Construir a Design Matrix $X$

### O que o código faz

```python
X = []
X.append(np.ones_like(x))   # coluna de 1s
X.append(x)                  # coluna de x
X = np.array(X).T
```

Constrói a **Design Matrix** — uma matriz onde cada **linha** é um ponto de dados e cada **coluna** é uma feature (incluindo o intercepto).

Resultado:
```
X = [[ 1, -7],
     [ 1, -4],
     [ 1, -2],
     [ 1,  0],
     [ 1,  2]]
```

---

### Por que existe a coluna de 1s?

O modelo linear que queremos ajustar é:

$$y_i = a_0 \cdot 1 + a_1 \cdot x_i$$

onde $a_0$ é o **intercepto** (bias) e $a_1$ é a **inclinação** (slope). Em forma matricial:

$$\mathbf{y} = X \cdot \mathbf{a} = \begin{bmatrix} 1 & x_1 \\ 1 & x_2 \\ \vdots & \vdots \\ 1 & x_N \end{bmatrix} \begin{bmatrix} a_0 \\ a_1 \end{bmatrix}$$

A coluna de 1s é necessária para que a multiplicação matricial $X \cdot \mathbf{a}$ produza o termo $a_0$ automaticamente. Sem ela, o modelo seria forçado a passar pela origem ($y = a_1 x$, sem intercepto).

---

### Como as ferramentas funcionam

**`np.ones_like(x)`**  
Cria um array de 1s com o mesmo shape e dtype de `x`. É equivalente a `np.ones(len(x))`, mas mais seguro: se `x` mudar de tamanho, a coluna de 1s muda junto automaticamente.

**`np.array(X).T`**  
O `np.array(X)` converte a lista de listas para uma matriz — neste ponto cada lista vira uma **linha**. O `.T` faz a **transposta**, virando cada lista em uma **coluna**. Resultado: shape `(N, 2)` onde N é o número de amostras.

```
Antes do .T:  shape (2, 5)  — 2 linhas (features), 5 colunas (amostras)
Depois do .T: shape (5, 2)  — 5 linhas (amostras), 2 colunas (features)
```

**Convenção universal em ML:** a Design Matrix tem shape `(N, p)` onde N = número de amostras e p = número de features (incluindo intercepto).

### Como proceder: forma mais direta com `np.column_stack`

O código original usa lista + append + array + transposta. Uma forma mais direta e legível:

```python
X = np.column_stack([np.ones_like(x), x])
# Equivalente, mas mais conciso e fácil de expandir para mais features:
# X = np.column_stack([np.ones_like(x), x, x**2, np.sin(x)])
```

In [ ]:
# Reproduzindo o código original passo a passo
X_lista = []
X_lista.append(np.ones_like(x))  # passo 1: coluna de 1s
X_lista.append(x)                 # passo 2: coluna de x

print("Antes de np.array().T (lista de listas):")
print(np.array(X_lista))
print(f"  shape: {np.array(X_lista).shape}  ← (2 features, 5 amostras)")

X = np.array(X_lista).T
print("\nDepois do .T (Design Matrix final):")
print(X)
print(f"  shape: {X.shape}  ← (5 amostras, 2 features)")

print("\n--- Forma alternativa mais concisa: np.column_stack ---")
X_alt = np.column_stack([np.ones_like(x), x])
print(X_alt)
print(f"  shape: {X_alt.shape}  ← idêntico ao anterior")
print(f"  São iguais? {np.array_equal(X, X_alt)}")

print("\nInterpretação de cada elemento X[i, j]:")
print("  X[i, 0] = 1.0  → feature do intercepto (sempre 1)")
print("  X[i, 1] = x_i  → valor real de x para a amostra i")
print(f"\n  Ex: X[2, :] = {X[2]}  → amostra x=-2: (1 × a₀) + (-2 × a₁)")

---

## Passo 3 — Montar as Normal Equations

### O que o código faz

```python
XTX = X.T @ X
XTy = X.T @ y
```

Calcula os dois componentes das **Normal Equations** — o sistema de equações cuja solução é o vetor de coeficientes $\mathbf{a}$ que minimiza o MSE.

---

### A matemática por trás: de onde vêm as Normal Equations?

O objetivo da Linear Regression é minimizar o **Sum of Squared Errors (SSE)**:

$$\text{SSE}(\mathbf{a}) = \|\mathbf{y} - X\mathbf{a}\|^2 = (\mathbf{y} - X\mathbf{a})^T(\mathbf{y} - X\mathbf{a})$$

Para encontrar o mínimo, derivamos em relação a $\mathbf{a}$ e igualamos a zero:

$$\frac{\partial \text{SSE}}{\partial \mathbf{a}} = -2X^T(\mathbf{y} - X\mathbf{a}) = 0$$

Reorganizando:

$$\boxed{X^TX \cdot \mathbf{a} = X^T\mathbf{y}}$$

Estas são as **Normal Equations**. São um sistema linear $A \cdot \mathbf{a} = b$ onde:
- $A = X^TX$ → matriz quadrada de shape `(p, p)`
- $b = X^T\mathbf{y}$ → vetor de shape `(p,)`
- $\mathbf{a}$ → o que queremos encontrar

---

### Como a ferramenta funciona: o operador `@`

Em Python, `@` é o operador de **multiplicação matricial** (matrix multiply), introduzido no Python 3.5.

```python
A @ B   # multiplicação matricial: equivale a np.matmul(A, B)
A * B   # multiplicação elemento a elemento (NÃO é multiplicação matricial!)
```

**Regra de shape para `@`:**  
Para `A @ B` funcionar, o número de colunas de `A` deve ser igual ao número de linhas de `B`:
```
A.shape = (m, n)  e  B.shape = (n, k)  →  (A @ B).shape = (m, k)
```

No nosso caso:
```
X.T  shape = (2, 5)
X    shape = (5, 2)
X.T @ X  →  shape (2, 2)   ← matriz quadrada p×p

X.T  shape = (2, 5)
y    shape = (5,)
X.T @ y  →  shape (2,)     ← vetor p
```

In [ ]:
XTX = X.T @ X
XTy = X.T @ y

print("=== Normal Equations: X^T X · a = X^T y ===")
print()
print("X^T X (shape:", XTX.shape, "):")
print(XTX)
print()
print("X^T y (shape:", XTy.shape, "):")
print(XTy)

print()
print("--- Verificação manual de X^T X ---")
print("X^T X[0,0] = soma de 1²  = N =", XTX[0, 0])
print("X^T X[0,1] = soma de 1·xᵢ = soma(x) =", XTX[0, 1], "(deve ser", x.sum(), ")")
print("X^T X[1,1] = soma de xᵢ² =", XTX[1, 1], "(deve ser", (x**2).sum(), ")")
print()
print("X^T y[0] = soma de 1·yᵢ = soma(y) =", XTy[0], "(deve ser", y.sum(), ")")
print("X^T y[1] = soma de xᵢ·yᵢ         =", XTy[1], "(deve ser", (x*y).sum(), ")")

print()
print("--- Atenção: @ vs * ---")
exemplo_a = np.array([[1, 2], [3, 4]])
exemplo_b = np.array([[5, 6], [7, 8]])
print("A @ B (multiplicação matricial):")
print(exemplo_a @ exemplo_b)
print("A * B (elemento a elemento — NÃO é multiplicação matricial!):")
print(exemplo_a * exemplo_b)

---

## Passo 4 — Resolver o sistema: `np.linalg.solve`

### O que o código faz

```python
a = np.linalg.solve(XTX, XTy)
```

Resolve o sistema linear $X^TX \cdot \mathbf{a} = X^T\mathbf{y}$ para encontrar o vetor de coeficientes $\mathbf{a}$.

Resultado: `a = [0.25, -0.25]`, ou seja:

$$\hat{y} = 0.25 \cdot 1 + (-0.25) \cdot x = 0.25 - 0.25x$$

---

### Como a ferramenta funciona: `np.linalg.solve`

`np.linalg.solve(A, b)` resolve o sistema $A \cdot x = b$ para $x$.

**Internamente**, a função usa **LU decomposition** (decomposição LU) — muito mais estável e eficiente do que calcular a inversa diretamente.

**Regra de ouro:** **nunca use** `np.linalg.inv(A) @ b` para resolver sistemas lineares. Calcular a inversa é:
- Mais lento
- Numericamente instável para matrizes mal condicionadas
- `np.linalg.solve` é sempre a escolha correta

```python
# ✗ NÃO faça isso:
a = np.linalg.inv(XTX) @ XTy   # lento e instável

# ✓ FAÇA isso:
a = np.linalg.solve(XTX, XTy)  # rápido e estável
```

---

### Quando `np.linalg.solve` falha?

A função lança `LinAlgError` se a matriz `A` for **singular** (não invertível), o que acontece quando:
- Duas features são linearmente dependentes (ex: `x` e `2x` ao mesmo tempo)
- Você tem mais features do que amostras ($p > N$) — sistema subdeterminado

Nestes casos, use **Ridge Regression** (regularização L2), que garante que `XTX + λI` é sempre invertível:

```python
lam = 0.01   # regularização
a_ridge = np.linalg.solve(XTX + lam * np.eye(XTX.shape[0]), XTy)
```

---

### Alternativas e quando usar cada uma

| Método | Quando usar |
|--------|-------------|
| `np.linalg.solve(XTX, XTy)` | Caso padrão: $N > p$, features independentes |
| `np.linalg.lstsq(X, y)` | Sistema sobredeterminado ou subdeterminado; mais robusto |
| `sklearn.linear_model.Ridge` | Quando há risco de multicolinearidade ou overfitting |
| `sklearn.linear_model.LinearRegression` | Produção: cuida de edge cases automaticamente |

In [ ]:
# Solução com np.linalg.solve (método do notebook original)
a = np.linalg.solve(XTX, XTy)

print("=== Coeficientes encontrados ===")
print(f"  a = {a}")
print(f"  a[0] = {a[0]:+.4f}  ← intercepto (a₀): o valor de y quando x=0")
print(f"  a[1] = {a[1]:+.4f}  ← inclinação (a₁): variação de y para cada +1 em x")
print(f"\n  Modelo: ŷ = {a[0]:+.4f} + ({a[1]:+.4f}) · x")
print(f"          ŷ = {a[0]:.2f} - {abs(a[1]):.2f}·x")

print()
print("--- Verificação: np.linalg.solve vs np.linalg.lstsq ---")
a_lstsq, residuals, rank, sv = np.linalg.lstsq(X, y, rcond=None)
print(f"  solve:  a = {a}")
print(f"  lstsq:  a = {a_lstsq}")
print(f"  Iguais? {np.allclose(a, a_lstsq)}")

print()
print("--- Verificar o residuo SSE ---")
y_pred_treino = X @ a
residuos = y - y_pred_treino
SSE = np.sum(residuos**2)
MSE = SSE / len(y)
print(f"  Previsões no treino: {y_pred_treino.round(3)}")
print(f"  Resíduos:           {residuos.round(3)}")
print(f"  SSE = {SSE:.4f}")
print(f"  MSE = {MSE:.4f}")

print()
print("--- O que acontece se tentarmos inverter (não recomendado) ---")
a_inv = np.linalg.inv(XTX) @ XTy
print(f"  Via inv(): {a_inv}  (mesmo resultado aqui, mas instável em geral)")

print()
print("--- Demonstração: Ridge resolve o caso singular ---")
X_sing = np.column_stack([np.ones_like(x), x, x])  # coluna x duplicada (singular!)
XTX_sing = X_sing.T @ X_sing
print(f"  Determinante de X^T X singular: {np.linalg.det(XTX_sing):.2e}  ← praticamente zero")
lam = 0.1
a_ridge = np.linalg.solve(XTX_sing + lam * np.eye(3), X_sing.T @ y)
print(f"  Ridge ainda resolve: a = {a_ridge.round(4)}")

---

## Passo 5 — Gerar previsões e plotar o resultado

### O que o código faz

```python
x_g = np.linspace(-7, 2, 100)
y_pred = a[0] + a[1] * x_g

import matplotlib.pyplot as plt
plt.plot(x_g, y_pred)
plt.scatter(x, y, c='r')
```

Cria 100 pontos igualmente espaçados entre -7 e 2 para traçar a reta suavemente, calcula as previsões do modelo para esses pontos, e plota a reta + os dados originais.

---

### Como as ferramentas funcionam

**`np.linspace(start, stop, num)`**  
Gera `num` valores igualmente espaçados entre `start` e `stop` (inclusive em ambos os extremos).  
Usado para criar um eixo suave para plotagem — sem `linspace`, a reta ficaria com apenas 5 pontos (os dados), parecendo segmentos de reta quebrados.

```python
np.linspace(0, 1, 5)   # → [0.0, 0.25, 0.5, 0.75, 1.0]
np.arange(0, 1, 0.25)  # → [0.0, 0.25, 0.5, 0.75]  (não inclui o stop!)
```

**`plt.plot(x, y)`** — plota uma **linha** conectando os pontos  
**`plt.scatter(x, y)`** — plota **pontos individuais** (sem conectar)

| Parâmetro | O que controla | Exemplo |
|-----------|---------------|---------|
| `c=` ou `color=` | Cor | `c='red'`, `color='tab:blue'` |
| `lw=` | Espessura da linha (linewidth) | `lw=2` |
| `ls=` | Estilo da linha (linestyle) | `ls='--'` (tracejada) |
| `s=` | Tamanho dos pontos (scatter) | `s=80` |
| `label=` | Legenda | `label="Modelo"` |
| `zorder=` | Camada de renderização | maior = na frente |

---

### Por que usar `np.linspace` em vez de calcular só nos dados de treino?

O modelo aprende com os 5 pontos de treino, mas a reta de regressão é definida para **qualquer valor de x**. `np.linspace` cria uma grade fina de pontos para visualizar essa reta contínua. Em geral, 100 pontos é suficiente para parecer suave.

### Como proceder: boas práticas de visualização

```python
# Sempre use fig, ax para maior controle:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_g, y_pred, label="Modelo", lw=2)
ax.scatter(x, y, c='red', s=80, label="Dados", zorder=3)
ax.set_title("Linear Regression")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
```

In [ ]:
# Gerar pontos para a reta
x_g = np.linspace(-7, 2, 100)
y_pred = a[0] + a[1] * x_g

print(f"x_g: {len(x_g)} pontos de {x_g[0]} a {x_g[-1]}")
print(f"y_pred nos extremos: {y_pred[0]:.3f} (x=-7) e {y_pred[-1]:.3f} (x=2)")

# Plot reproduzindo exatamente o notebook original
plt.figure(figsize=(8, 5))
plt.plot(x_g, y_pred, label=f"ŷ = {a[0]:.2f} + ({a[1]:.2f})x", lw=2)
plt.scatter(x, y, c='r', s=80, zorder=3, label="dados originais")
plt.title("Linear Regression — Resultado")
plt.xlabel("x"); plt.ylabel("y")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Versão aprimorada: mostrar também os resíduos
y_pred_treino = a[0] + a[1] * x

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(x_g, y_pred, "b-", lw=2, label=f"ŷ = {a[0]:.2f} - {abs(a[1]):.2f}x")
ax.scatter(x, y, c='red', s=80, zorder=3, label="dados")
for xi, yi, yh in zip(x, y, y_pred_treino):
    ax.plot([xi, xi], [yi, yh], "k:", lw=1.2, alpha=0.6)
ax.set_title("Reta ajustada + resíduos (linhas pontilhadas)")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
residuos = y - y_pred_treino
ax.stem(x, residuos, basefmt=" ")
ax.axhline(0, color='k', lw=1)
ax.set_title("Resíduos por ponto (y − ŷ)")
ax.set_xlabel("x"); ax.set_ylabel("Resíduo")
ax.grid(alpha=0.3)
for xi, ri in zip(x, residuos):
    ax.annotate(f"{ri:.3f}", (xi, ri), textcoords="offset points",
                xytext=(5, 5), fontsize=9)

plt.suptitle("Visualização completa: Linear Regression", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

---

## Recapitulação: o pipeline completo em 10 linhas

O notebook original faz algo muito poderoso em pouquíssimas linhas. Veja abaixo o mesmo resultado escrito de forma compacta e comentada:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Dados
x = np.array([-7, -4, -2, 0, 2])
y = np.array([ 1,  2,  2, 0, -1])

# 2. Design Matrix [1 | x]
X = np.column_stack([np.ones_like(x), x])

# 3. Normal Equations
XTX = X.T @ X
XTy = X.T @ y

# 4. Resolver para os coeficientes
a = np.linalg.solve(XTX, XTy)

# 5. Plotar
x_g = np.linspace(x.min(), x.max(), 100)
plt.figure(figsize=(7, 4))
plt.plot(x_g, a[0] + a[1]*x_g, lw=2, label=f"ŷ = {a[0]:.3f} + ({a[1]:.3f})x")
plt.scatter(x, y, c='r', s=80, zorder=3, label="dados")
plt.title("Linear Regression do zero"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Coeficientes: intercepto = {a[0]:.4f}, inclinação = {a[1]:.4f}")

---

## Guia de ferramentas: referência rápida

### NumPy — operações usadas

| Ferramenta | Sintaxe | O que faz |
|------------|---------|----------|
| Criar array | `np.array([1, 2, 3])` | Converte lista para array NumPy |
| Array de 1s | `np.ones_like(x)` | Array de 1s com mesmo shape de `x` |
| Empilhar colunas | `np.column_stack([a, b])` | Une arrays como colunas de uma matriz |
| Transposta | `A.T` | Inverte linhas e colunas |
| Multiplicação matricial | `A @ B` | Produto matricial (não elemento a elemento) |
| Resolver sistema linear | `np.linalg.solve(A, b)` | Encontra x em A·x = b (via LU, estável) |
| Grid de pontos | `np.linspace(a, b, n)` | n pontos igualmente espaçados entre a e b |
| Norma vetorial | `np.linalg.norm(v)` | ‖v‖ (comprimento do vetor) |
| Verificar igualdade | `np.allclose(a, b)` | Compara arrays com tolerância numérica |

### Matplotlib — operações usadas

| Ferramenta | Sintaxe | O que faz |
|------------|---------|----------|
| Linha contínua | `plt.plot(x, y)` | Conecta pontos com uma linha |
| Pontos dispersos | `plt.scatter(x, y)` | Plota pontos individuais sem conexão |
| Linha horizontal | `plt.axhline(val)` | Linha horizontal em y=val |
| Título | `plt.title("texto")` | Adiciona título ao gráfico |
| Eixos | `plt.xlabel`, `plt.ylabel` | Rótulos dos eixos |
| Legenda | `plt.legend()` | Mostra a legenda com os labels |
| Grid | `plt.grid(alpha=0.3)` | Grade de fundo semitransparente |
| Múltiplos gráficos | `fig, axes = plt.subplots(1, 2)` | Cria figura com subplots lado a lado |

---

## Quando adaptar: checklist para novos problemas

Ao aplicar Linear Regression do zero em um novo dataset, siga esta sequência:

**1. Verificar os dados**
```python
print(x.shape, y.shape)   # devem ter o mesmo N
print(np.isnan(x).any())  # verificar NaN
```

**2. Construir a Design Matrix certa**
```python
# Regressão linear simples:
X = np.column_stack([np.ones_like(x), x])
# Regressão polinomial grau 2:
X = np.column_stack([np.ones_like(x), x, x**2])
# Múltiplas features:
X = np.column_stack([np.ones(N), x1, x2, x3])
```

**3. Verificar se o sistema tem solução única**
```python
cond = np.linalg.cond(XTX)
print(f"Condition number: {cond:.1e}")   # >1e12 → problema de singularidade
```

**4. Resolver e validar**
```python
a = np.linalg.solve(X.T @ X, X.T @ y)
y_pred = X @ a
MSE = np.mean((y - y_pred)**2)
R2  = 1 - np.sum((y - y_pred)**2) / np.sum((y - y.mean())**2)
print(f"MSE = {MSE:.4f}, R² = {R2:.4f}")
```

**5. Plotar sempre: dados + reta + resíduos**

---

## Exercícios para consolidar

1. **Mude os dados:** adicione um ponto `(4, -3)` ao dataset. Os coeficientes mudam? Em qual direção?
2. **Regressão quadrática:** crie uma Design Matrix com colunas `[1, x, x²]` e resolva o sistema. Compare a reta com a parábola ajustada.
3. **Compare com scikit-learn:** use `sklearn.linear_model.LinearRegression` e verifique que os coeficientes são idênticos ao `np.linalg.solve`.
4. **Condition number:** duplique a coluna `x` na Design Matrix e observe o que acontece com o condition number e com `np.linalg.solve`.
5. **R² score:** implemente manualmente o R² e interprete: o modelo é bom para este dataset?